In [1]:
import torch
import torch.nn.functional as F

# custom utils
from trnbl import TrainingManager
from trnbl.loggers.local import LocalLogger


from attention_motifs.ae import AttnAEConfig, AttnAE

f:\projects\attention-motifs\.venv\Lib\site-packages\trnbl\loggers\base.py:17: UserWarning: GPUtil not available: No module named 'GPUtil'
  warnings.warn(f"GPUtil not available: {e}")


In [2]:
# magic autoreload
%load_ext autoreload
%autoreload 2

In [3]:
config: AttnAEConfig = AttnAEConfig(
	latent_dim=64,
)

In [4]:
model: AttnAE = AttnAE(config)

model

AttnAE(
  (encoder): Encoder(
    (conv): Sequential(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(16, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
    )
    (linear_prepool): Sequential(
      (0): Linear(in_features=64, out_features=128, bias=True)
      (1): ReLU()
    )
    (linear_postpool): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=64, bias=True)
      (3): ReLU()
    )
  )
  (decoder): Decoder(
    (linear_postunpool): Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=128, bias=True)
      (3): ReLU()
    )
    (linear_preunpool): Sequential(
      (0): Linear(in_features=128, out_features=128, bias=True)
      (1): ReLU()
    )
    (conv): Sequential(
      (0): ConvTranspose2d(64, 16, kernel_size=(

In [5]:
train_loader: torch.utils.data.DataLoader
val_loader: torch.utils.data.DataLoader | None = None
num_epochs: int = 100
learning_rate: float = 1e-3
recon_weight: float = 1.0
contrast_weight: float = 1.0
project_name: str = "contrastive-ae"
checkpoint_interval: str = "1/10 run"
eval_interval: str = "1K samples"
device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = model.to(device)
optimizer: torch.optim.Optimizer = model.config.optimizer(
	model.parameters(),
	lr=learning_rate,
)

print(model.zanj_model_config.serialize())

# setup logger
logger: LocalLogger = LocalLogger(
	project=project_name,
	metric_names=[
		"train/loss",
		"train/recon_loss",
		"train/contrast_loss",
		"val/loss",
		"val/recon_loss",
		"val/contrast_loss",
	],
	train_config=dict(
		model_config=model.zanj_model_config.serialize(),
		learning_rate=learning_rate,
		recon_weight=recon_weight,
		contrast_weight=contrast_weight,
	),
)

{'__format__': 'AttnAEConfig(SerializableDataclass)', 'latent_dim': 64, 'in_channels': 1, 'conv_encoder': [{'__format__': 'Conv2DConfig(SerializableDataclass)', 'channels': 16, 'kernel_size': 3, 'stride': 1, 'padding': 1}, {'__format__': 'Conv2DConfig(SerializableDataclass)', 'channels': 64, 'kernel_size': 3, 'stride': 1, 'padding': 1}], 'mlp_prepool': [128], 'mlp_postpool': [128, 64], 'activation': 'ReLU', 'margin': 1.0, 'optimizer': 'Adam'}
# starting logger with id hc1fa1-250128_0000-zoteqe


In [27]:
from attention_motifs.dataset.dataset import CollectedAttentionPatternDataloader

train_loader_dataset = CollectedAttentionPatternDataloader.read("../data/activations/pile_5")
val_loader_dataset = CollectedAttentionPatternDataloader.read("../data/activations/pile_5_val")


In [28]:
print(train_loader_dataset)

{
  "model_names": [
    "pythia-14m",
    "gpt2-small",
    "meta-llama/Llama-3.2-1B"
  ],
  "dataset_shapes_summary": [
    "(11936, 192, 192)",
    "(408, 256, 256)",
    "(824, 64, 64)",
    "(680, 128, 128)",
    "(576, 512, 512)"
  ],
  "n_datasets": 5,
  "n_total_samples": 14424,
  "n_ctx_stats": {
    "total_items": 14424,
    "n_keys": 5,
    "mode": 192,
    "mean": 196.25956738768718,
    "std": 72.91070113289837,
    "min": 64,
    "q1": 192.0,
    "median": 192.0,
    "q3": 192.0,
    "max": 512
  },
  "config": {
    "__format__": "APGenerationConfig(SerializableDataclass)",
    "prompts_config": {
      "__format__": "PromptDatasetConfig(SerializableDataclass)",
      "name": "pile_5",
      "source_path": "../data/pile_5.jsonl",
      "source_info": {
        "source_path": "../data/pile_5.jsonl"
      },
      "char_len_min": 64,
      "char_len_max": 1024
    },
    "model_names": [
      "pythia-14m",
      "gpt2-small",
      "meta-llama/Llama-3.2-1B"
    ],
    "to

In [7]:
def evaluation_step(model: AttnAE) -> dict[str, float]:
	"""Evaluate model on validation set"""
	if val_loader is None:
		return {}

	model.eval()
	val_metrics = {"val/loss": 0.0, "val/recon_loss": 0.0, "val/contrast_loss": 0.0}

	with torch.no_grad():
		for batch_idx, (x, index_tuple) in enumerate(val_loader):
			x = x.to(device)
			index_tuple = tuple(i.to(device) for i in index_tuple)

			x_recon, z = model(x)
			recon_loss = F.mse_loss(x_recon, x)

			batch_size = x.size(0)
			z1 = z.repeat_interleave(batch_size, dim=0)
			z2 = z.repeat(batch_size, 1)
			idx1 = tuple(i.repeat_interleave(batch_size) for i in index_tuple)
			idx2 = tuple(i.repeat(batch_size) for i in index_tuple)
			contrast_loss = model.contrastive_loss(z1, z2, idx1, idx2)

			total_loss = recon_weight * recon_loss + contrast_weight * contrast_loss

			val_metrics["val/loss"] += total_loss.item()
			val_metrics["val/recon_loss"] += recon_loss.item()
			val_metrics["val/contrast_loss"] += contrast_loss.item()

	for k in val_metrics:
		val_metrics[k] /= len(val_loader)

	model.train()
	return val_metrics

In [8]:
with TrainingManager(
	model=model,
	logger=logger,
	evals={
		eval_interval: evaluation_step,
	}.items(),
	checkpoint_interval=checkpoint_interval,
) as tr:
	for epoch in tr.epoch_loop(range(num_epochs)):
		for x, index_tuple in tr.batch_loop(train_loader):
			x = x.to(device)
			index_tuple = tuple(i.to(device) for i in index_tuple)

			optimizer.zero_grad()
			x_recon, z = model(x)

			# reconstruction loss
			recon_loss = F.mse_loss(x_recon, x)

			# contrastive loss using all pairs in batch
			batch_size = x.size(0)
			z1 = z.repeat_interleave(batch_size, dim=0)
			z2 = z.repeat(batch_size, 1)
			idx1 = tuple(i.repeat_interleave(batch_size) for i in index_tuple)
			idx2 = tuple(i.repeat(batch_size) for i in index_tuple)
			contrast_loss = model.contrastive_loss(z1, z2, idx1, idx2)

			# combined loss and backward pass
			total_loss = recon_weight * recon_loss + contrast_weight * contrast_loss
			total_loss.backward()
			optimizer.step()

			# log metrics
			tr.batch_update(
				samples=len(x),
				**{
					"train/loss": total_loss.item(),
					"train/recon_loss": recon_loss.item(),
					"train/contrast_loss": contrast_loss.item(),
				},
			)

# starting training manager initialization


training run:   0%|          | 0/100 [00:00<?, ? epochs/s]

# ERROR: object of type 'generator' has no len()


# closing logger


TypeError: object of type 'generator' has no len()